# English Streaming ASR — Kaggle: runnable foundation

This is the first notebook in the E2E execution sequence. It contains only the commands that can run now without model downloads or external decoder integration. It implements Sections 1 and 2 of `docx/steps.md`.

**Already completed:** environment setup, Phase 0 smoke test, Phase 3 demo generation, and Phase 5 demo OSD.

**Next notebooks:** `notebooks/11_whisperrt_baseline.ipynb`, `notebooks/12_adaptive_e2e_surt.ipynb`, and `notebooks/13_surt_training_evaluation.ipynb`.

In [ ]:
# Clone the repository
!git clone https://github.com/manzourali/English-Streaming-ASR.git

## Notebook rules

Run cells from top to bottom. The repository must be located at `/kaggle/working/English-Streaming-ASR`. Do not rerun the demo generation cell unless you intentionally want to replace the current small fixture dataset.

In [ ]:
%cd /kaggle/working/English-Streaming-ASR
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'src').is_dir(), f'Run this in the cloned repository, not {ROOT}'
print(f'Repository root: {ROOT}')
print('Runbook:', ROOT / 'docx/steps.md')

## Completed-run reference

The Kaggle environment previously reported Python 3.13.15, a Tesla T4 GPU, CUDA 12.8, and successful project imports. The Phase 0 smoke test succeeded. The demo generator created 4 mixtures in each split, and demo OSD measured precision 1.0000, recall 0.6364, F1 0.7778, and OSD RTF 0.00866. These are integration results, not final research results.

In [ ]:
import sys
import os
import platform
import subprocess

# ── Python & OS ──────────────────────────────────────────────────────────────
print("=" * 55)
print("ENVIRONMENT CHECK")
print("=" * 55)

print(f"\n[Python]")
print(f"  Version      : {sys.version}")
print(f"  Executable   : {sys.executable}")

print(f"\n[OS]")
print(f"  System       : {platform.system()} {platform.release()}")
print(f"  Machine      : {platform.machine()}")

# ── Detect runtime environment ───────────────────────────────────────────────
print(f"\n[Runtime Environment]")

in_colab  = 'google.colab' in sys.modules or os.path.exists('/content')
in_kaggle = os.path.exists('/kaggle')

if in_colab:
    print("  Platform     : Google Colab")
elif in_kaggle:
    print("  Platform     : Kaggle")
else:
    print("  Platform     : Local / Other")

# ── GPU check ────────────────────────────────────────────────────────────────
print(f"\n[GPU]")
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        for line in result.stdout.strip().split('\n'):
            print(f"  {line}")
    else:
        print("  No NVIDIA GPU detected")
except FileNotFoundError:
    print("  nvidia-smi not found — no GPU or not NVIDIA")

# ── Key packages ─────────────────────────────────────────────────────────────
print(f"\n[Key Packages]")
packages = [
    'torch', 'tensorflow', 'numpy', 'pandas',
    'transformers', 'sklearn', 'cv2', 'PIL',
    'torchaudio', 'soundfile', 'librosa'
]

for pkg in packages:
    try:
        mod = __import__(pkg)
        version = getattr(mod, '__version__', 'unknown')
        print(f"  {pkg:<15}: {version}")
    except ImportError:
        print(f"  {pkg:<15}: NOT installed")

# ── Disk & RAM ────────────────────────────────────────────────────────────────
print(f"\n[Resources]")
try:
    import shutil
    total, used, free = shutil.disk_usage('/')
    print(f"  Disk free    : {free  // (1024**3)} GB")
    print(f"  Disk total   : {total // (1024**3)} GB")
except Exception as e:
    print(f"  Disk info unavailable: {e}")

try:
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith(('MemTotal', 'MemAvailable')):
                print(f"  {line.strip()}")
except FileNotFoundError:
    print("  /proc/meminfo not available")

# ── Working directory & repo ─────────────────────────────────────────────────
print(f"\n[Working Directory]")
print(f"  CWD          : {os.getcwd()}")
print(f"  Files        : {os.listdir('.')[:10]}")   # show first 10 entries

print("\n" + "=" * 55)
print("CHECK COMPLETE")
print("=" * 55)


In [ ]:
!python -m pip install -q -r requirements.txt
# install pyproject.toml requirements
!python -m pip install -q -e ".[audio,phase1,phase2]"

In [ ]:
!python scripts/check_environment.py
!python scripts/smoke_test.py --phase 0 --config configs/base.yaml

In [ ]:
!python scripts/generate_overlap.py --config configs/overlap_dataset.yaml --demo
!python scripts/evaluate_osd.py --config configs/overlap_detection.yaml --max-samples 4

# 1. Immediate commands: validation and VAD

These commands are safe to run now and do not download an ASR model.

## 1.1 Validate the Phase 3 manifests for Phase 4

This checks that all generated mixtures and JSONL records can be read by the overlap-ASR benchmark. It does **not** load WhisperRT. Expected result: `status: validated_only`, `num_examples: 12`, and `model_run: NOT_RUN`.

In [ ]:
!python scripts/run_overlap_baseline.py \
  --config configs/overlap_baseline.yaml \
  --validate-only

## 1.2 Run the standalone streaming VAD demo

The first command runs VAD without external audio. The second checks the project's real WebRTC VAD integration. Neither produces a labeled VAD accuracy score.

In [ ]:
!python scripts/run_vad.py --config configs/streaming_vad.yaml
!python scripts/smoke_test.py \
  --phase 2 \
  --real-vad \
  --config configs/streaming_vad.yaml

## 1.3 Validate adaptive routing and frozen evaluation configurations

Validation confirms YAML/configuration consistency only. It does not load WhisperRT, SURT, or create E2E scores.

In [ ]:
!python scripts/run_adaptive.py \
  --config configs/adaptive.yaml \
  --validate-only

!python scripts/evaluate.py \
  --config configs/evaluation.yaml \
  --validate-only

# 2. Persist this session

Kaggle's working disk is temporary. Run this section before stopping the session, then create a private Kaggle Dataset from the bundle folder.

## 2.1 Inspect both output roots

The overlap generator writes under the repository. Some runtime scripts detect Kaggle and write under `/kaggle/working/outputs`.

In [ ]:
!find /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap -maxdepth 3 -type f | sort
!find /kaggle/working/outputs -maxdepth 3 -type f 2>/dev/null | sort

## 2.2 Create the portable Phase 3 + Phase 5 artifact bundle

After this cell, use Kaggle's **Save Version** / dataset creation flow to persist `/kaggle/working/asr_phase3_phase5_artifact`. Do not add large generated audio or model weights to Git.

In [ ]:
!mkdir -p /kaggle/working/asr_phase3_phase5_artifact
!cp -a /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap /kaggle/working/asr_phase3_phase5_artifact/
!cp -a /kaggle/working/outputs/phase5 /kaggle/working/asr_phase3_phase5_artifact/ 2>/dev/null || true
!cp -a /kaggle/working/outputs/reports /kaggle/working/asr_phase3_phase5_artifact/ 2>/dev/null || true
!cp configs/overlap_dataset.yaml configs/overlap_detection.yaml /kaggle/working/asr_phase3_phase5_artifact/
!git rev-parse HEAD | tee /kaggle/working/asr_phase3_phase5_artifact/git_commit.txt
!find /kaggle/working/asr_phase3_phase5_artifact -maxdepth 3 -type f | sort

## Continue

Open `notebooks/11_whisperrt_baseline.ipynb` only after the bundle is saved. It contains the required Hugging Face authorization and WhisperRT-runtime compatibility gate.